#Reto IA: Telefonica challenge - Extracción de festures y creación de los CSV

Este documento contiene el flujo principal para transformar los archivos de audio crudos en datos estructurados listos para entrenar y testear los modelos.

Fases del proceso:
*  Carga de datos: Lectura de los audios de entrenamiento y validación.
*  Extracción y enriquecimiento: Cálculo de las features de audio y asignación de metadatos mediante una función automatizada.
*  Exportación: Guardado de los resultados definitivos en un archivo CSV, optimizado para su ingesta inmediata en un DataFrame.

Este pipeline se ha ejecutado en múltiples ocasiones a medida que se han ido incorporando nuevos audios para el análisi


# 0. Importación de librerías, connexión con drive y cargar áudios

In [ ]:
import numpy as np
import pandas as pd
import librosa
import os
from scipy import stats
from tqdm import tqdm
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
TIPO_DATASET = "sintetico"
ruta_carpeta = "/content/drive/MyDrive/Reto_Telefonica/audios_dataset_sint/Acento_Chileno_F-M/"

In [ ]:
print("Carpeta configurada:")
print(ruta_carpeta)

print("\n¿Existe la carpeta?:", os.path.exists(ruta_carpeta))

if os.path.exists(ruta_carpeta):
    print("\nPrimeros archivos encontrados:")
    print(os.listdir(ruta_carpeta)[:10])

Carpeta configurada:
/content/drive/MyDrive/Reto_Telefonica/audios_dataset_sint/Acento_Chileno_F-M/

¿Existe la carpeta?: True

Primeros archivos encontrados:
['CycleGAN-clf_05223_01102038735-clf_03397_019712.wav', 'CycleGAN-clf_05223_01948250535-clf_04310_009487.wav', 'CycleGAN-clf_05223_00086574876-clf_04310_016292.wav', 'CycleGAN-clf_05223_00571992422-clf_07049_007720.wav', 'CycleGAN-clf_05223_01578311399-clf_04310_000891.wav', 'CycleGAN-clf_05223_00759208930-clf_04310_020744.wav', 'CycleGAN-clf_05223_01601629897-clf_03397_014183.wav', 'CycleGAN-clf_05223_00571992422-clf_04310_014822.wav', 'CycleGAN-clf_05223_00759208930-clf_03397_002525.wav', 'CycleGAN-clf_05223_01601629897-clf_04310_004879.wav']


In [ ]:
#Cargar audios
archivos = os.listdir(ruta_carpeta)
print(f"Se detectaron {len(archivos)} archivos en la carpeta.\n")

for archivo in archivos:
    if archivo.lower().endswith(".wav"):
        ruta_audio = os.path.join(ruta_carpeta, archivo)

        y, sr = librosa.load(ruta_audio, sr=16000)

        print(f"{archivo} cargado | duración: {len(y)/sr:.2f} segundos")

Se detectaron 500 archivos en la carpeta.

CycleGAN-clf_05223_01102038735-clf_03397_019712.wav cargado | duración: 5.73 segundos
CycleGAN-clf_05223_01948250535-clf_04310_009487.wav cargado | duración: 7.61 segundos
CycleGAN-clf_05223_00086574876-clf_04310_016292.wav cargado | duración: 5.39 segundos
CycleGAN-clf_05223_00571992422-clf_07049_007720.wav cargado | duración: 4.45 segundos
CycleGAN-clf_05223_01578311399-clf_04310_000891.wav cargado | duración: 7.53 segundos
CycleGAN-clf_05223_00759208930-clf_04310_020744.wav cargado | duración: 5.31 segundos
CycleGAN-clf_05223_01601629897-clf_03397_014183.wav cargado | duración: 9.15 segundos
CycleGAN-clf_05223_00571992422-clf_04310_014822.wav cargado | duración: 4.45 segundos
CycleGAN-clf_05223_00759208930-clf_03397_002525.wav cargado | duración: 5.31 segundos
CycleGAN-clf_05223_01601629897-clf_04310_004879.wav cargado | duración: 9.15 segundos
CycleGAN-clf_05223_01692372116-clf_03397_002178.wav cargado | duración: 5.15 segundos
CycleGAN-cl

# 1.0 Definición de funciones para extraer features

In [ ]:
#Generamos función para extraer sus atributos de unáudio a través de su nombre
def extraer_etiquetas(nombre_audio, modo="auto"):
    """
    Extrae etiquetas binarias desde el nombre del audio.

    Devuelve un diccionario con:
    - label: 0 real / 1 sintético
    - genero_f: 1 mujer / 0 hombre
    - nacionalidad: colombiano / chileno / argentino / peruano
    - modelo generador: CycleGAN, Diff, StarGAN, TTS-Dif, TTS-StarGAN, TTS
    """

    nombre = nombre_audio.lower()

    etiquetas = {
        # Label principal
        "label": 0,

        # 🔥 Género (solo una columna)
        "genero_f": 0,

        # Nacionalidad
        "colombiano": 0,
        "chileno": 0,
        "argentino": 0,
        "peruano":0,

        # Modelo generador
        "modelo_cyclegan": 0,
        "modelo_diff": 0,
        "modelo_stargan": 0,
        "modelo_tts_dif": 0,
        "modelo_tts_stargan": 0,
        "modelo_tts": 0
    }

    # -----------------------------
    # 1. Label principal
    # -----------------------------
    if modo == "real":
        etiquetas["label"] = 0
    elif modo == "sintetico":
        etiquetas["label"] = 1
    else:
        etiquetas["label"] = 1 if "-" in nombre_audio else 0

    # -----------------------------
    # 2. Nacionalidad + género
    # -----------------------------
    if "com" in nombre:
        etiquetas["colombiano"] = 1
        etiquetas["genero_f"] = 0

    elif "cof" in nombre:
        etiquetas["colombiano"] = 1
        etiquetas["genero_f"] = 1

    elif "clm" in nombre:
        etiquetas["chileno"] = 1
        etiquetas["genero_f"] = 0

    elif "clf" in nombre:
        etiquetas["chileno"] = 1
        etiquetas["genero_f"] = 1

    elif "arm" in nombre:
        etiquetas["argentino"] = 1
        etiquetas["genero_f"] = 0

    elif "arf" in nombre:
        etiquetas["argentino"] = 1
        etiquetas["genero_f"] = 1

    elif "pem" in nombre:
        etiquetas["peruano"] = 1
        etiquetas["genero_f"] = 0

    elif "pef" in nombre:
        etiquetas["peruano"] = 1
        etiquetas["genero_f"] = 1

    # -----------------------------
    # 3. Modelo generador
    # -----------------------------
    nombre_original = nombre_audio  # mantener mayúsculas

    if "TTS-StarGAN" in nombre_original:
        etiquetas["modelo_tts_stargan"] = 1
    elif "TTS-Dif" in nombre_original:
        etiquetas["modelo_tts_dif"] = 1
    elif "CycleGAN" in nombre_original:
        etiquetas["modelo_cyclegan"] = 1
    elif "StarGAN" in nombre_original:
        etiquetas["modelo_stargan"] = 1
    elif "Diff" in nombre_original:
        etiquetas["modelo_diff"] = 1
    elif "TTS" in nombre_original:
        etiquetas["modelo_tts"] = 1

    return etiquetas

In [ ]:
#Función para ejecutar la extración de etiquetas y guardarlo
def generar_df_etiquetas(ruta_carpeta, tipo_dataset="auto"):
    """
    Genera un DataFrame con múltiples etiquetas:
    - id_audio: identificador secuencial (1, 2, 3, ...)
    - archivo: nombre original del archivo
    - label: real (0) vs sintético (1)
    - genero_f: mujer (1), hombre (0)
    - nacionalidad: colombiano / chileno / argentino / peruano
    - modelo generador
    """

    print(f"Explorando carpeta: {ruta_carpeta}")
    datos = []

    contador_id = 1

    for archivo in sorted(os.listdir(ruta_carpeta)):
        if archivo.lower().endswith(".wav"):
            nombre_audio = archivo.replace(".wav", "")

            etiquetas = extraer_etiquetas(nombre_audio, modo=tipo_dataset)

            fila = {
                "id_audio": contador_id,
                "archivo": archivo
            }

            fila.update(etiquetas)
            datos.append(fila)

            contador_id += 1

    df_etiquetas = pd.DataFrame(datos)

    print(f"DataFrame creado con {len(df_etiquetas)} registros.")
    print(f"Número de columnas: {df_etiquetas.shape[1]}")

    return df_etiquetas

In [ ]:
df_labels = generar_df_etiquetas(ruta_carpeta, tipo_dataset=TIPO_DATASET)
df_labels.head()

Explorando carpeta: /content/drive/MyDrive/Reto_Telefonica/audios_dataset_sint/Acento_Chileno_F-M/
DataFrame creado con 500 registros.
Número de columnas: 14


,id_audio,archivo,label,genero_f,colombiano,chileno,argentino,peruano,modelo_cyclegan,modelo_diff,modelo_stargan,modelo_tts_dif,modelo_tts_stargan,modelo_tts
0,1,CycleGAN-clf_05223_00086574876-clf_03397_01336...,1,1,0,1,0,0,1,0,0,0,0,0
1,2,CycleGAN-clf_05223_00086574876-clf_04310_01629...,1,1,0,1,0,0,1,0,0,0,0,0
2,3,CycleGAN-clf_05223_00086574876-clf_07049_00147...,1,1,0,1,0,0,1,0,0,0,0,0
3,4,CycleGAN-clf_05223_00086574876-clf_09697_00965...,1,1,0,1,0,0,1,0,0,0,0,0
4,5,CycleGAN-clf_05223_00571992422-clf_03397_00831...,1,1,0,1,0,0,1,0,0,0,0,0


## 1.1 Extracción de features

Además de la extracción de etiquetas y metadatos —como la naturaleza real o sintética del audio, el género del locutor y el modelo generador utilizado—, el siguiente paso crítico es la extracción de las características acústicas (features). La selección y el procesamiento de estas métricas se fundamentan en la metodología descrita por Mael Fabien*, la cual detalla qué atributos de la señal de voz son los más idóneos para transformar el audio en datos estructurados y optimizados para modelos de Machine Learning


* https://maelfabien.github.io/machinelearning/Speech9/#7-mel-frequency-cepstral-differential-coefficients


A continuación, se detallan las características extraídas por cada archivo de audio:

1. Metadatos Temporales y de Ritmo
*  Duración (duracion_seg): El tiempo total del audio en segundos (tras eliminar los silencios iniciales y finales).
*  Tempo (tempo): Una estimación del ritmo del audio medido en pulsos por minuto (BPM). Ayuda a detectar si la cadencia del habla es natural.

2. Energía y Amplitud (Dominio del Tiempo)
*  ZCR (zcr - Zero Crossing Rate): Mide la tasa a la que la señal cambia de signo (de positivo a negativo). Es fundamental para identificar sonidos ruidosos o consonantes fricativas (como la "s" o la "f"), donde las IAs suelen cometer errores.
*  RMS (rms) y RMSE Manual (rmse_manual): Miden la energía de la raíz cuadrada media de la señal. En términos prácticos, representan el volumen percibido o la intensidad del audio. Se extrae tanto de forma automatizada (vía librosa) como mediante cálculo matemático directo.

3. Coeficientes Cepstrales (La "Huella Dactilar" de la Voz)
*  MFCC (mfcc_1 a mfcc_13): Los Coeficientes Cepstrales en las Frecuencias de Mel. Son 13 bandas que describen la "textura" o el timbre de la voz, modelando cómo el oído humano percibe el sonido.
*  Delta MFCC (delta_mfcc): La primera derivada (velocidad de cambio) de los MFCC. Indica cómo cambia el timbre de la voz de un instante a otro (transiciones entre fonemas).
*  Delta-Delta MFCC (delta2_mfcc): La segunda derivada (aceleración) de los MFCC. Captura variaciones aún más finas y rápidas en el habla, muy difíciles de replicar perfectamente por un modelo sintético.

4. Características Espectrales (Dominio de la Frecuencia)
*  Centroide Espectral (centroid): Representa el "centro de masa" del espectro. Determina si un sonido se percibe como "brillante" (frecuencias altas) u "opaco" (frecuencias bajas).
*  Ancho de Banda Espectral (bandwidth): Mide la dispersión de las frecuencias alrededor del centroide. Un ancho de banda artificialmente estrecho puede ser un claro indicio de compresión o generación por IA.
*  Contraste Espectral (contrast): Calcula la diferencia de amplitud entre los picos y los valles del espectro de frecuencias. Ayuda a distinguir sonidos armónicos de ruidos de fondo.
*  Planitud Espectral (flatness): Mide qué tan similar al "ruido blanco" es un sonido. Valores altos indican ruido (sin tono claro), mientras que valores bajos indican un sonido tonal (como una vocal clara).
*  Rolloff Espectral (rolloff): Representa la frecuencia por debajo de la cual se concentra el 85% de la energía del audio. Es muy útil para detectar si un modelo generativo ha aplicado cortes artificiales en las frecuencias más agudas.


In [ ]:
#Función de extracción de features
def extraer_features(ruta_audio, sr_objetivo=16000, n_mfcc=13):
    """
    Extrae features de un audio y devuelve un diccionario.
    """

    y, sr = librosa.load(ruta_audio, sr=sr_objetivo)

    # Recorte de silencios
    y, _ = librosa.effects.trim(y)

    if len(y) < 512:
        return None

    features = {}

    # -------------------
    # META
    # -------------------
    features["duracion_seg"] = len(y) / sr

    # -------------------
    # ZCR
    # -------------------
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    features.update(resumir_feature(zcr, "zcr"))

    # -------------------
    # RMS (librosa)
    # -------------------
    rms = librosa.feature.rms(y=y)[0]
    features.update(resumir_feature(rms, "rms"))

    # -------------------
    # RMSE manual
    # -------------------
    rmse_manual = np.sqrt(np.mean(y**2))
    features["rmse_manual"] = rmse_manual

    # -------------------
    # TEMPO
    # -------------------
    tempo = librosa.beat.tempo(y=y, sr=sr)[0]
    features["tempo"] = tempo

    # -------------------
    # MFCC
    # -------------------
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    for i in range(n_mfcc):
        features.update(resumir_feature(mfcc[i], f"mfcc_{i+1}"))

    # -------------------
    # DELTA MFCC
    # -------------------
    delta_mfcc = librosa.feature.delta(mfcc)
    for i in range(n_mfcc):
        features.update(resumir_feature(delta_mfcc[i], f"delta_mfcc_{i+1}"))

    # -------------------
    # DELTA-DELTA MFCC
    # -------------------
    delta2_mfcc = librosa.feature.delta(mfcc, order=2)
    for i in range(n_mfcc):
        features.update(resumir_feature(delta2_mfcc[i], f"delta2_mfcc_{i+1}"))

    # -------------------
    # SPECTRAL FEATURES
    # -------------------
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    features.update(resumir_feature(centroid, "centroid"))

    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    features.update(resumir_feature(bandwidth, "bandwidth"))

    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    for i in range(contrast.shape[0]):
        features.update(resumir_feature(contrast[i], f"contrast_{i+1}"))

    flatness = librosa.feature.spectral_flatness(y=y)[0]
    features.update(resumir_feature(flatness, "flatness"))

    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    features.update(resumir_feature(rolloff, "rolloff"))

    # -------------------
    # PRINT FINAL
    # -------------------
    print(f"🎧 {os.path.basename(ruta_audio)}")
    print(f"   - Features extraídas: {len(features)}")
    print(f"   - Tempo: {tempo:.2f}")
    print(f"   - RMSE manual: {rmse_manual:.5f}")

    return features

Dado que el sonido es una señal dinámica que cambia a lo largo del tiempo, extraer un único valor estático no proporciona información suficiente. Por ello, la gran mayoría de estas métricas las procesamos mediante una función de resumen que extrae 11 estadísticas descriptivas para cada variable temporal: media, desviación estándar, mínimo, máximo, mediana, cuartiles (Q1 y Q3), asimetría (skewness), curtosis, moda y rango intercuartílico (IQR). Esto nos permite capturar no solo el tono promedio del audio, sino también la micro-variabilidad y las posibles anomalías matemáticas que suelen dejar los algoritmos de IA al generar voces falsas.

In [ ]:
#Función generación estadísticas de las features
def resumir_feature(vector, prefijo, debug=False):
    vector = np.asarray(vector).astype(float)

    if len(vector) == 0:
        return {}

    resultado = {
        f"{prefijo}_mean": np.mean(vector),
        f"{prefijo}_std": np.std(vector),
        f"{prefijo}_min": np.min(vector),
        f"{prefijo}_max": np.max(vector),
        f"{prefijo}_median": np.median(vector),
        f"{prefijo}_q1": np.quantile(vector, 0.25),
        f"{prefijo}_q3": np.quantile(vector, 0.75),
        f"{prefijo}_skew": stats.skew(vector),
        f"{prefijo}_kurtosis": stats.kurtosis(vector),
        f"{prefijo}_mode": stats.mode(vector, keepdims=True)[0][0],
        f"{prefijo}_iqr": stats.iqr(vector)
    }

    if debug:
        print(f"\n🔍 Feature: {prefijo}")
        print(f"   Total valores generados: {len(resultado)}")
        for k, v in list(resultado.items())[:3]:
            print(f"   {k}: {v:.4f}")

    return resultado

#2.0 Ejecución de las funciones

In [ ]:
registros = []

archivos_wav = sorted([a for a in os.listdir(ruta_carpeta) if a.lower().endswith(".wav")])

for idx, archivo in enumerate(tqdm(archivos_wav), start=1):
    ruta_audio = os.path.join(ruta_carpeta, archivo)
    nombre_audio = archivo.replace(".wav", "")

    try:
        etiquetas = extraer_etiquetas(nombre_audio, modo=TIPO_DATASET)
        feats = extraer_features(ruta_audio)

        if feats is None:
            print(f"Audio omitido por ser demasiado corto: {archivo}")
            continue

        fila = {
            "id_audio": idx,
            "archivo": archivo
        }

        fila.update(etiquetas)
        fila.update(feats)

        registros.append(fila)

    except Exception as e:
        print(f"Error procesando {archivo}: {e}")

  0%|          | 0/500 [00:00<?, ?it/s]/tmp/ipykernel_2955/2322890922.py:43: FutureWarning: librosa.beat.tempo
	This function was moved to 'librosa.feature.rhythm.tempo' in librosa version 0.10.0.
	This alias will be removed in librosa version 1.0.
  tempo = librosa.beat.tempo(y=y, sr=sr)[0]
  0%|          | 1/500 [00:17<2:26:42, 17.64s/it]

🎧 CycleGAN-clf_05223_00086574876-clf_03397_013360.wav
   - Features extraídas: 575
   - Tempo: 144.23
   - RMSE manual: 0.09616


  0%|          | 2/500 [00:17<1:01:29,  7.41s/it]

🎧 CycleGAN-clf_05223_00086574876-clf_04310_016292.wav
   - Features extraídas: 575
   - Tempo: 144.23
   - RMSE manual: 0.06964


  1%|          | 3/500 [00:18<34:16,  4.14s/it]  

...

🎧 TTS-clm_01208_01096922678.wav
   - Features extraídas: 575
   - Tempo: 125.00
   - RMSE manual: 0.12051
🎧 TTS-clm_01208_01207410495.wav
   - Features extraídas: 575
   - Tempo: 75.00
   - RMSE manual: 0.12103


100%|█████████▉| 499/500 [02:39<00:00,  4.74it/s]

🎧 TTS-clm_01208_01216467857.wav
   - Features extraídas: 575
   - Tempo: 125.00
   - RMSE manual: 0.10225


100%|██████████| 500/500 [02:39<00:00,  3.13it/s]

🎧 TTS-clm_01208_01229211467.wav
   - Features extraídas: 575
   - Tempo: 104.17
   - RMSE manual: 0.11146


In [ ]:
print("\n Creando DataFrame...")

df = pd.DataFrame(registros)

print(" DataFrame creado")
print(f"   - Filas: {df.shape[0]}")
print(f"   - Columnas: {df.shape[1]}")

df.head()


 Creando DataFrame...
 DataFrame creado
   - Filas: 500
   - Columnas: 589


,id_audio,archivo,label,genero_f,colombiano,chileno,argentino,peruano,modelo_cyclegan,modelo_diff,...,rolloff_std,rolloff_min,rolloff_max,rolloff_median,rolloff_q1,rolloff_q3,rolloff_skew,rolloff_kurtosis,rolloff_mode,rolloff_iqr
0,1,CycleGAN-clf_05223_00086574876-clf_03397_01336...,1,1,0,1,0,0,1,0,...,1737.794563,171.8750,7273.4375,3640.62500,1898.437500,4929.687500,-0.015848,-1.125790,2757.8125,3031.250000
1,2,CycleGAN-clf_05223_00086574876-clf_04310_01629...,1,1,0,1,0,0,1,0,...,1919.135148,187.5000,7343.7500,4339.84375,2068.359375,5525.390625,-0.192997,-1.358087,1187.5000,3457.031250
2,3,CycleGAN-clf_05223_00086574876-clf_07049_00147...,1,1,0,1,0,0,1,0,...,1952.102029,171.8750,7289.0625,4453.12500,2132.812500,5656.250000,-0.239998,-1.343159,2351.5625,3523.437500
3,4,CycleGAN-clf_05223_00086574876-clf_09697_00965...,1,1,0,1,0,0,1,0,...,1761.820821,187.5000,7312.5000,4066.40625,2281.250000,5138.671875,-0.160017,-1.171392,1398.4375,2857.421875
4,5,CycleGAN-clf_05223_00571992422-clf_03397_00831...,1,1,0,1,0,0,1,0,...,1921.594961,195.3125,7109.3750,3015.62500,1787.109375,5351.562500,0.170794,-1.353723,5351.5625,3564.453125


In [ ]:
#Revisar columnas
print("\n Información de columnas:")

print(f"Total columnas: {len(df.columns)}")

print("\nPrimeras 20 columnas:")
print(df.columns[:20])

print("\nÚltimas 20 columnas:")
print(df.columns[-20:])


 Información de columnas:
Total columnas: 589

Primeras 20 columnas:
Index(['id_audio', 'archivo', 'label', 'genero_f', 'colombiano', 'chileno',
       'argentino', 'peruano', 'modelo_cyclegan', 'modelo_diff',
       'modelo_stargan', 'modelo_tts_dif', 'modelo_tts_stargan', 'modelo_tts',
       'duracion_seg', 'zcr_mean', 'zcr_std', 'zcr_min', 'zcr_max',
       'zcr_median'],
      dtype='object')

Últimas 20 columnas:
Index(['flatness_min', 'flatness_max', 'flatness_median', 'flatness_q1',
       'flatness_q3', 'flatness_skew', 'flatness_kurtosis', 'flatness_mode',
       'flatness_iqr', 'rolloff_mean', 'rolloff_std', 'rolloff_min',
       'rolloff_max', 'rolloff_median', 'rolloff_q1', 'rolloff_q3',
       'rolloff_skew', 'rolloff_kurtosis', 'rolloff_mode', 'rolloff_iqr'],
      dtype='object')


In [ ]:
#Revisar información general
print("\n Información general del DataFrame:")
df.info()


 Información general del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Columns: 589 entries, id_audio to rolloff_iqr
dtypes: float32(1), float64(574), int64(13), object(1)
memory usage: 2.2+ MB


In [ ]:
print("\n Estadísticas descriptivas:")
df.describe()


 Estadísticas descriptivas:


,id_audio,label,genero_f,colombiano,chileno,argentino,peruano,modelo_cyclegan,modelo_diff,modelo_stargan,...,rolloff_std,rolloff_min,rolloff_max,rolloff_median,rolloff_q1,rolloff_q3,rolloff_skew,rolloff_kurtosis,rolloff_mode,rolloff_iqr
count,500.000000,500.0,500.000000,500.0,500.0,500.0,500.0,500.000000,500.000000,500.000000,...,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000
mean,250.500000,1.0,0.472000,0.0,1.0,0.0,0.0,0.180000,0.320000,0.180000,...,1754.080831,429.046875,7099.250000,3611.132812,2125.988281,4886.449219,0.128658,-0.712670,3070.125000,2760.460938
std,144.481833,0.0,0.499715,0.0,0.0,0.0,0.0,0.384572,0.466943,0.384572,...,273.119124,321.899904,400.596396,919.802351,545.370701,827.208143,0.596289,0.760836,1819.720512,892.600991
min,1.000000,1.0,0.000000,0.0,1.0,0.0,0.0,0.000000,0.000000,0.000000,...,1135.049808,93.750000,4929.687500,1812.500000,1070.312500,2449.218750,-1.163694,-1.762974,109.375000,746.093750
25%,125.750000,1.0,0.000000,0.0,1.0,0.0,0.0,0.000000,0.000000,0.000000,...,1549.268871,187.500000,6873.046875,2768.554688,1732.421875,4458.496094,-0.332500,-1.300267,1505.859375,2156.250000
50%,250.500000,1.0,0.000000,0.0,1.0,0.0,0.0,0.000000,0.000000,0.000000,...,1697.935043,320.312500,7171.875000,3585.937500,2002.929688,4878.906250,0.025874,-0.844605,2625.000000,2743.164062
75%,375.250000,1.0,1.000000,0.0,1.0,0.0,0.0,0.000000,1.000000,0.000000,...,1956.876954,539.062500,7382.812500,4411.132812,2428.222656,5445.312500,0.543444,-0.337515,4634.765625,3312.988281
max,500.000000,1.0,1.000000,0.0,1.0,0.0,0.0,1.000000,1.000000,1.000000,...,2493.051114,1835.937500,7828.125000,5679.687500,4855.468750,7103.515625,1.666475,2.522596,7515.625000,5162.109375


In [ ]:
#Revisamos una fila
print("\n Inspección de una fila completa:")

fila = df.iloc[0]

print(fila)
print("\nTotal de valores en esta fila:", len(fila))


 Inspección de una fila completa:
id_audio                                                            1
archivo             CycleGAN-clf_05223_00086574876-clf_03397_01336...
label                                                               1
genero_f                                                            1
colombiano                                                          0
                                          ...                        
rolloff_q3                                                  4929.6875
rolloff_skew                                                -0.015848
rolloff_kurtosis                                             -1.12579
rolloff_mode                                                2757.8125
rolloff_iqr                                                   3031.25
Name: 0, Length: 589, dtype: object

Total de valores en esta fila: 589


#3.0 Guardado y descarga del dataset

In [ ]:
#Guardamos el dataset
print("\n Guardando dataset...")

nombre_csv = f"dataset_features_{TIPO_DATASET}.csv"
ruta_salida_csv = f"/content/drive/MyDrive/Reto_Telefonica/{nombre_csv}"

df.to_csv(ruta_salida_csv, index=False)

print(" Dataset guardado correctamente")
print(f" Ruta: {ruta_salida_csv}")
print(f" Shape final: {df.shape}")


 Guardando dataset...
 Dataset guardado correctamente
 Ruta: /content/drive/MyDrive/Reto_Telefonica/dataset_features_sintetico.csv
 Shape final: (500, 589)


In [ ]:
#Descargamos el dataset
print("\n Preparando descarga...")

from google.colab import files
files.download(ruta_salida_csv)

print(" Descarga iniciada")


 Preparando descarga...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Descarga iniciada
